# Chapter 1

Add your content here.

In [1]:
import numpy as np

class ToyMambaSSD:
	def __init__(self, seq_len, decay_rate=0.9):
		"""
		Initialize simple scalar SSM parameters.
		A: Decay rate (scalar)
		B: Input scale (scalar)
		C: Output scale (scalar)
		"""
		self.L = seq_len
		self.A = decay_rate  # Simple scalar decay
		self.B = 2.0  # Arbitrary input projection
		self.C = 1.5  # Arbitrary output projection

	def recurrent_mode(self, x):
		"""
		Mode 1: The RNN approach (Sequential)
		h_t = A * h_{t-1} + B * x_t
		y_t = C * h_t
		"""
		x = np.asarray(x)
		if x.shape[0] != self.L:
			raise ValueError("Input length must match seq_len")
		h = 0.0  # Initial state
		y_preds = []

		for t in range(self.L):
			xt = x[t]
			# 1. Update State
			h = self.A * h + self.B * xt
			# 2. Compute Output
			yt = self.C * h
			y_preds.append(yt)

		return np.array(y_preds)

	def attention_mode(self, x):
		"""
		Mode 2: The SSD Matrix approach (Parallel)
		Y = (Mask * (C x B_transpose)) X
		"""
		x = np.asarray(x)
		if x.shape[0] != self.L:
			raise ValueError("Input length must match seq_len")
		# 1. Construct the Materialized Matrix M
		M = np.zeros((self.L, self.L))

		# We fill M based on M_ij = C * A^(i-j) * B
		for i in range(self.L):
			for j in range(i + 1):  # Lower triangular only
				power_of_A = self.A ** (i - j)
				M[i, j] = self.C * power_of_A * self.B

		# 2. Perform one giant Matrix Multiplication
		# This is what Tensor Cores love!
		y_preds = M @ x

		return y_preds

# ---Verification--

# 1. Create a random input sequence (e.g., a time series signal)
SEQUENCE_LENGTH = 10
x_input = np.random.randn(SEQUENCE_LENGTH)

# 2. Instantiate Model
model = ToyMambaSSD(SEQUENCE_LENGTH, decay_rate=0.5)

# 3. Run both modes
y_rnn = model.recurrent_mode(x_input)
y_attn = model.attention_mode(x_input)

# 4. Compare
print("Input:", np.round(x_input, 2))
print("RNN Output: ", np.round(y_rnn, 2))
print("Attention Output:", np.round(y_attn, 2))

# Check mathematical equivalence (within floating point tolerance)
difference = np.linalg.norm(y_rnn - y_attn)
if difference < 1e-10:
	print(f"\nSUCCESS: The Dualism holds! Difference: {difference}")
else:
	print(f"\nFAILURE: Difference is {difference}")

Input: [ 0.42  0.19 -1.45 -0.74  0.75 -1.85 -1.1   0.31 -1.14 -0.27]
RNN Output:  [ 1.26  1.19 -3.76 -4.09  0.21 -5.46 -6.02 -2.06 -4.46 -3.05]
Attention Output: [ 1.26  1.19 -3.76 -4.09  0.21 -5.46 -6.02 -2.06 -4.46 -3.05]

SUCCESS: The Dualism holds! Difference: 7.021666937153402e-16
